In [51]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path
import json
from cleaning.profile import profile
import cleaning.checks as chk 
from dataclasses import asdict   # only if the script itself converts results to a table



RAW = Path('C:/repositaries/4.personal/EEA_interview/data/raw')

source_code= 'demo_gind'

#----
file_path = Path(RAW, f'{source_code}')


#read the parquet file
df = pd.read_parquet(file_path/'df.parquet')
lookups = json.load(open(file_path / 'lookup.json', encoding='utf-8'))




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
df["value"].describe()

profile(df, lookups)


shape: (23800, 5)

dtypes:
freq            str
indic_de        str
geo             str
time            str
value       float64

type of time variable:
Annual: 23800
Quarterly: 0
Monthly: 0
Daily: 0

missing share per column:
freq        0.000
indic_de    0.000
geo         0.000
time        0.000
value       0.248

constant columns: ['freq']

value summary:
count    1.788700e+04
mean     2.427162e+06
std      9.369512e+06
min     -8.847810e+05
25%      1.040000e+01
50%      1.833100e+04
75%      3.569885e+05
max      1.436669e+08

lookup vs data (code counts/values in dimension):
  freq: lookup=1, in data=1
  indic_de: lookup=28, in data=28
  geo: lookup=50, in data=50
  time: lookup=17, in data=17


In [53]:
results = [
    chk.unique_key(df, key=["indic_de", "geo", "time"]),
    chk.codes_in_lookup(df, lookups),
    chk.non_negative(df),
    chk.numeric_dtypes(df),
    chk.missing_share(df, cols=["indic_de", "geo", "time"], max_share=0.0, severity="error"),
    chk.missing_share(df, cols=["value"], max_share=0.5, severity="info"),
    chk.constant_columns(df),
    chk.dims_match_lookup(df, lookups),
]

report = pd.DataFrame(asdict(r) for r in results)

results

[CheckResult(check='unique_key', severity='error', passed=True, n_issues=0, detail="key=['indic_de', 'geo', 'time']"),
 CheckResult(check='codes_in_lookup', severity='error', passed=True, n_issues=0, detail='{}'),
 CheckResult(check='non_negative', severity='error', passed=False, n_issues=1666, detail='value'),
 CheckResult(check='numeric_dtypes', severity='warning', passed=False, n_issues=1, detail="{'time': 'dtype=str, not convertible=0'}"),
 CheckResult(check='missing_share', severity='error', passed=True, n_issues=indic_de    0.0
 geo         0.0
 time        0.0
 dtype: float64, detail="max_share=0.0; all={'indic_de': 0.0, 'geo': 0.0, 'time': 0.0}; over={}"),
 CheckResult(check='missing_share', severity='info', passed=True, n_issues=value    0.248
 dtype: float64, detail="max_share=0.5; all={'value': 0.248}; over={}"),
 CheckResult(check='constant_columns', severity='info', passed=False, n_issues=1, detail="['freq']"),
 CheckResult(check='dims_match_lookup', severity='error', pass